[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-7_executed.ipynb)

---

# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 7: The Human-AI Symbiotic Partnership (The Centaur Scheduler)

This notebook presents a small, transparent prototype of a human-AI workforce scheduling partnership. The scheduler generates three candidate schedules using different rules: cost-oriented, employee-satisfaction-oriented, and robustness-oriented. A manager can compare the candidates, select one, and request a shift change. The system reports the consequences of that feedback and records the interaction.

### Inputs and outputs
- **Inputs:** employee skills, hourly rates, maximum working hours, preferred shifts, hourly staffing demand, and manager priorities.
- **Outputs:** candidate schedules, coverage and cost measures, employee-preference scores, robustness measures, explanations, and the consequences of manager feedback.
- **Time representation:** the operating horizon and shift endpoints are measured in minutes. Demand is specified for consecutive one-hour periods, and employee hourly rates are measured in currency units per hour.

### Modeling assumptions and scope
- A schedule assigns each employee at most one continuous shift. Shifts are half-open intervals: an employee covers a demand period when the shift includes that period's entire interval.
- The supplied demand values are required staffing levels, not forecasts. The candidate-generation rules use this fixed demand pattern.
- The prototype enforces maximum working hours and reports whether each demand period is covered. It does not model breaks, overtime, labor laws, skill-specific coverage, uncertain demand, or partial-period shifts.
- The candidate generators are transparent heuristics, not exact optimization algorithms. Their labels describe the rule used to construct a candidate; they do not imply a global optimality guarantee.
- The interaction history records manager choices and feedback. It is an auditable record, not a trained learning model; this example does not claim that the system improves automatically over time.

### Partnership workflow
1. Generate one candidate for each scheduling objective.
2. Explain each candidate using measurable schedule properties.
3. Let a manager compare candidates using their stated priorities.
4. Apply an explicit shift-change request and show its effects.
5. Record the selection and feedback for review.

The purpose is to make the human-AI decision process inspectable. A manager remains responsible for contextual judgments that this simplified model does not represent.

## Scheduling model and measures

Let employees be indexed by $i \in E$ and demand periods by $t \in T$. Period $t$ occupies the interval $[a_t,b_t)$ and requires $d_t$ employees. Employee $i$ has hourly rate $r_i$, maximum working time $H_i$ hours, and a scheduled shift $[s_i,e_i)$. An employee covers period $t$ when $s_i \leq a_t$ and $e_i \geq b_t$.

The number of employees covering period $t$ is

$$
c_t = \sum_{i \in E} \mathbf{1}\{s_i \leq a_t \text{ and } e_i \geq b_t\},
$$

where $\mathbf{1}\{\cdot\}$ is 1 when its condition is true and 0 otherwise. The period is covered when $c_t \geq d_t$. The code calculates this count in `coverage_by_period` and checks the condition in `evaluate_schedule`.

The scheduled labor cost is

$$
C = \sum_{i \in E} r_i \frac{e_i-s_i}{60},
$$

where shift duration is in minutes, so division by 60 converts it to hours. This measure is calculated by `evaluate_schedule`; it is wage cost for scheduled time, not a full payroll-cost model.

Employee-preference satisfaction is represented by the fraction of scheduled employees whose shift category matches their first listed preference. It is a simple, interpretable proxy, not a psychological or survey-based measure. Robustness is represented by the minimum coverage surplus across periods, $\min_{t \in T}(c_t-d_t)$. A positive surplus means that every period has at least one employee above its stated requirement; a negative value identifies a coverage shortfall.

These measures describe and compare the candidates. The notebook does not combine them into a single mathematical optimum: the manager's priorities are used to rank the candidates transparently, and the manager makes the final selection.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")
np.random.seed(42)

## Data structures

The data classes below keep employee information, manager preferences, and schedule recommendations explicit. A `ScheduleOption` stores the proposed shifts alongside the measures and explanation needed for a human review. The optional feedback fields distinguish the AI-generated proposal from a manager-refined schedule.

In [ ]:
@dataclass
class Employee:
    """Employee attributes used by the scheduling prototype."""

    id: int
    name: str
    skills: List[str]
    hourly_rate: float
    max_hours: int
    preferred_shifts: List[str] = field(default_factory=list)
    unavailable_periods: List[Tuple[float, float]] = field(default_factory=list)
    satisfaction_weight: float = 0.8
    experience_level: str = "intermediate"


@dataclass
class HumanManager:
    """Manager attributes used to support review and candidate ranking."""

    id: int
    name: str
    department: str
    experience_years: int
    decision_style: str
    priorities: Dict[str, float] = field(default_factory=dict)
    trust_in_ai: float = 0.7
    risk_tolerance: float = 0.5


@dataclass
class ScheduleOption:
    """A candidate schedule with measures and a human-readable explanation."""

    id: str
    name: str
    schedule: Dict[int, List[float]]
    objective_scores: Dict[str, float]
    explanation: str
    trade_offs: List[str]
    confidence_score: float
    human_feedback: Optional[Dict] = None
    refined_schedule: Optional[Dict[int, List[float]]] = None

## The Centaur Scheduler

The candidate-generation rules preserve the three-objective teaching structure. The cost-oriented rule assigns full-horizon shifts to the least expensive employees needed for peak demand. The satisfaction-oriented rule starts with employees' preferred shift patterns and extends selected shifts when necessary to cover demand. The robustness-oriented rule schedules as many employees as their hour limits allow, creating additional staffing capacity.

The measures make the trade-offs visible rather than hiding them inside a black-box score. `generate_schedule_options` constructs the candidates; `evaluate_schedule` checks coverage, maximum hours, cost, preference matching, and surplus staffing. These rules are intentionally small enough to inspect and are not guaranteed to find the best possible schedule.

In [ ]:
class CentaurScheduler:
    """Rule-based candidate generator for a human-reviewed schedule."""

    def __init__(
        self,
        employees: List[Employee],
        demand_pattern: List[float],
        cost_params: Dict,
        time_horizon: float = 480.0,
    ):
        if not employees:
            raise ValueError("At least one employee is required.")
        if not demand_pattern:
            raise ValueError("The demand pattern must contain at least one period.")
        if time_horizon <= 0:
            raise ValueError("The time horizon must be positive.")
        if any(demand < 0 for demand in demand_pattern):
            raise ValueError("Demand values must be non-negative.")
        if len({employee.id for employee in employees}) != len(employees):
            raise ValueError("Employee IDs must be unique.")

        self.employees = employees
        self.demand_pattern = demand_pattern
        self.cost_params = cost_params
        self.time_horizon = float(time_horizon)
        self.collaboration_history: List[Dict] = []
        self.preference_patterns: Dict[int, Dict] = {}
        self.successful_patterns: List[Dict] = []

    def _period_bounds(self, period_index: int) -> Tuple[float, float]:
        """Return the interval covered by one equally sized demand period."""
        period_length = self.time_horizon / len(self.demand_pattern)
        start = period_index * period_length
        return start, start + period_length

    def coverage_by_period(
        self, schedule: Dict[int, List[float]]
    ) -> List[int]:
        """Count employees covering each complete demand period."""
        coverage = []
        for period_index in range(len(self.demand_pattern)):
            period_start, period_end = self._period_bounds(period_index)
            count = sum(
                1
                for start, end in schedule.values()
                if start <= period_start and end >= period_end
            )
            coverage.append(count)
        return coverage

    def _shift_for_preference(self, employee: Employee) -> List[float]:
        """Create a feasible shift that follows the employee's first preference."""
        max_duration = min(self.time_horizon, employee.max_hours * 60.0)
        preference = (
            employee.preferred_shifts[0]
            if employee.preferred_shifts
            else "full_shift"
        )

        if preference == "morning":
            return [0.0, min(self.time_horizon / 2, max_duration)]
        if preference == "afternoon":
            duration = min(self.time_horizon / 2, max_duration)
            return [self.time_horizon - duration, self.time_horizon]
        if preference == "full_shift":
            start = (self.time_horizon - max_duration) / 2
            return [start, start + max_duration]
        raise ValueError(
            f"Unsupported preferred shift {preference!r} for "
            f"employee {employee.id}."
        )

    def evaluate_schedule(self, schedule: Dict[int, List[float]]) -> Dict[str, float]:
        """Measure schedule feasibility, cost, preference match, and surplus."""
        employee_by_id = {employee.id: employee for employee in self.employees}
        unknown_ids = set(schedule) - set(employee_by_id)
        if unknown_ids:
            raise ValueError(f"Unknown employee IDs in schedule: {unknown_ids}")

        for employee_id, interval in schedule.items():
            if len(interval) != 2:
                raise ValueError("Each shift must have a start and an end time.")
            start, end = interval
            employee = employee_by_id[employee_id]
            if start < 0 or end > self.time_horizon or end <= start:
                raise ValueError(f"Invalid shift interval for employee {employee_id}.")
            if end - start > employee.max_hours * 60.0:
                raise ValueError(
                    f"Employee {employee_id}'s shift exceeds their maximum hours."
                )

        coverage = self.coverage_by_period(schedule)
        covered_periods = sum(
            actual >= required
            for actual, required in zip(coverage, self.demand_pattern)
        )
        coverage_fraction = covered_periods / len(self.demand_pattern)
        coverage_surplus = min(
            actual - required
            for actual, required in zip(coverage, self.demand_pattern)
        )
        total_cost = sum(
            employee_by_id[employee_id].hourly_rate
            * (end - start)
            / 60.0
            for employee_id, (start, end) in schedule.items()
        )

        matched_preferences = 0
        for employee_id, (start, end) in schedule.items():
            employee = employee_by_id[employee_id]
            preference = (
                employee.preferred_shifts[0]
                if employee.preferred_shifts
                else "full_shift"
            )
            midpoint = self.time_horizon / 2
            if preference == "morning":
                matches = start == 0 and end <= midpoint
            elif preference == "afternoon":
                matches = start >= midpoint and end == self.time_horizon
            else:
                matches = start <= midpoint and end >= midpoint
            matched_preferences += int(matches)

        satisfaction = (
            matched_preferences / len(schedule) if schedule else 0.0
        )
        return {
            "cost": total_cost,
            "satisfaction": satisfaction,
            "robustness": float(coverage_surplus),
            "coverage_fraction": coverage_fraction,
            "covered_periods": float(covered_periods),
            "total_periods": float(len(self.demand_pattern)),
        }

    def _create_schedule_option(
        self,
        option_id: str,
        name: str,
        schedule: Dict[int, List[float]],
        manager: HumanManager,
    ) -> ScheduleOption:
        """Attach transparent measures and an explanation to a candidate."""
        scores = self.evaluate_schedule(schedule)
        coverage = self.coverage_by_period(schedule)
        shortfalls = [
            index
            for index, (actual, required) in enumerate(
                zip(coverage, self.demand_pattern)
            )
            if actual < required
        ]
        if shortfalls:
            coverage_note = (
                "Coverage shortfalls occur in period indices "
                f"{shortfalls}."
            )
        else:
            coverage_note = "All demand periods meet their stated staffing requirements."

        explanation = (
            f"Generated using the {option_id.replace('_', ' ')} rule. "
            f"Scheduled labor cost is {scores['cost']:.2f}; "
            f"the preference-match score is {scores['satisfaction']:.1%}; "
            f"the minimum coverage surplus is {scores['robustness']:.0f}. "
            f"{coverage_note} Manager priorities for this review: "
            f"{manager.priorities}."
        )
        trade_offs = [
            f"Scheduled labor cost: {scores['cost']:.2f}",
            f"Employee-preference match: {scores['satisfaction']:.1%}",
            f"Minimum coverage surplus: {scores['robustness']:.0f}",
        ]
        return ScheduleOption(
            id=option_id,
            name=name,
            schedule={key: value.copy() for key, value in schedule.items()},
            objective_scores=scores,
            explanation=explanation,
            trade_offs=trade_offs,
            confidence_score=scores["coverage_fraction"],
        )

    def generate_schedule_options(
        self, manager: HumanManager, n_options: int = 3
    ) -> List[ScheduleOption]:
        """Generate cost, satisfaction, and robustness candidates."""
        if n_options != 3:
            raise ValueError("This prototype generates exactly three objective options.")

        candidate_data = [
            (
                "cost_optimized",
                "Minimum Cost Schedule",
                self._optimize_for_cost(manager),
            ),
            (
                "satisfaction_maximized",
                "Maximum Employee Satisfaction",
                self._optimize_for_satisfaction(manager),
            ),
            (
                "robust",
                "Most Robust Schedule",
                self._optimize_for_robustness(manager),
            ),
        ]
        return [
            self._create_schedule_option(option_id, name, schedule, manager)
            for option_id, name, schedule in candidate_data
        ]

    def _optimize_for_cost(
        self, manager: HumanManager
    ) -> Dict[int, List[float]]:
        """Assign full-horizon shifts to the least expensive feasible staff."""
        del manager
        max_demand = int(max(self.demand_pattern))
        sorted_employees = sorted(
            self.employees, key=lambda employee: employee.hourly_rate
        )
        schedule = {}
        for employee in sorted_employees:
            if len(schedule) >= max_demand:
                break
            if employee.max_hours * 60.0 >= self.time_horizon:
                schedule[employee.id] = [0.0, self.time_horizon]
        return schedule

    def _optimize_for_satisfaction(
        self, manager: HumanManager
    ) -> Dict[int, List[float]]:
        """Start from preferred shifts, then extend shifts to cover demand."""
        del manager
        schedule = {
            employee.id: self._shift_for_preference(employee)
            for employee in sorted(
                self.employees,
                key=lambda item: item.satisfaction_weight,
                reverse=True,
            )
        }

        for period_index, required in enumerate(self.demand_pattern):
            period_start, period_end = self._period_bounds(period_index)
            while sum(
                start <= period_start and end >= period_end
                for start, end in schedule.values()
            ) < required:
                candidates = [
                    employee
                    for employee in self.employees
                    if schedule[employee.id] != [0.0, self.time_horizon]
                    and employee.max_hours * 60.0 >= self.time_horizon
                ]
                if not candidates:
                    break
                employee = min(
                    candidates, key=lambda item: item.satisfaction_weight
                )
                schedule[employee.id] = [0.0, self.time_horizon]
        return schedule

    def _optimize_for_robustness(
        self, manager: HumanManager
    ) -> Dict[int, List[float]]:
        """Schedule every employee for their maximum feasible horizon."""
        del manager
        schedule = {}
        for employee in self.employees:
            duration = min(self.time_horizon, employee.max_hours * 60.0)
            start = (self.time_horizon - duration) / 2
            schedule[employee.id] = [start, start + duration]
        return schedule

    def rank_options(
        self, options: List[ScheduleOption], manager: HumanManager
    ) -> List[Tuple[ScheduleOption, float]]:
        """Rank candidates by manager priorities after normalizing measures."""
        if not options:
            return []

        costs = [option.objective_scores["cost"] for option in options]
        min_cost, max_cost = min(costs), max(costs)
        ranked = []
        for option in options:
            scores = option.objective_scores
            cost_score = (
                1.0
                if max_cost == min_cost
                else (max_cost - scores["cost"]) / (max_cost - min_cost)
            )
            priority_score = (
                manager.priorities.get("cost", 0.0) * cost_score
                + manager.priorities.get("satisfaction", 0.0)
                * scores["satisfaction"]
                + manager.priorities.get("coverage", 0.0)
                * scores["coverage_fraction"]
                + manager.priorities.get("robustness", 0.0)
                * max(0.0, scores["robustness"])
            )
            ranked.append((option, priority_score))
        return sorted(ranked, key=lambda item: item[1], reverse=True)

    def apply_human_feedback(
        self,
        option: ScheduleOption,
        manager: HumanManager,
        feedback: Dict,
    ) -> ScheduleOption:
        """Apply requested shift categories and record their consequences."""
        overrides = feedback.get("shift_overrides", {})
        refined_schedule = {
            employee_id: interval.copy()
            for employee_id, interval in option.schedule.items()
        }
        employee_by_id = {employee.id: employee for employee in self.employees}

        for raw_employee_id, shift_name in overrides.items():
            employee_id = int(raw_employee_id)
            if employee_id not in employee_by_id:
                raise ValueError(f"Unknown employee ID in feedback: {employee_id}")
            if shift_name not in {"morning", "afternoon", "full_shift"}:
                raise ValueError(f"Unsupported shift category: {shift_name!r}")
            employee = employee_by_id[employee_id]
            original_preferences = employee.preferred_shifts
            employee.preferred_shifts = [shift_name]
            try:
                refined_schedule[employee_id] = self._shift_for_preference(employee)
            finally:
                employee.preferred_shifts = original_preferences

        scores = self.evaluate_schedule(refined_schedule)
        refined_option = self._create_schedule_option(
            f"{option.id}_human_refined",
            f"{option.name} (Human-Refined)",
            refined_schedule,
            manager,
        )
        refined_option.human_feedback = feedback.copy()
        refined_option.refined_schedule = {
            key: value.copy() for key, value in refined_schedule.items()
        }
        self.collaboration_history.append(
            {
                "manager_id": manager.id,
                "selected_option_id": option.id,
                "feedback": feedback.copy(),
                "result_scores": scores.copy(),
            }
        )
        self.preference_patterns.setdefault(manager.id, {}).update(
            {str(key): value for key, value in overrides.items()}
        )
        return refined_option

## Initialize the workforce and demand

The example describes an eight-hour operating horizon divided into eight one-hour periods. Demand rises to five employees at its peak. These values are inputs to the candidate generators and are not altered by the analysis.

In [ ]:
employees = [
    Employee(
        id=1,
        name="Alice",
        skills=["customer_service", "cashier"],
        hourly_rate=22,
        max_hours=8,
        preferred_shifts=["morning"],
        satisfaction_weight=0.9,
        experience_level="senior",
    ),
    Employee(
        id=2,
        name="Bob",
        skills=["customer_service", "stocking"],
        hourly_rate=20,
        max_hours=8,
        preferred_shifts=["afternoon"],
        satisfaction_weight=0.7,
        experience_level="intermediate",
    ),
    Employee(
        id=3,
        name="Charlie",
        skills=["cashier", "supervisor"],
        hourly_rate=24,
        max_hours=6,
        preferred_shifts=["full_shift"],
        satisfaction_weight=0.8,
        experience_level="senior",
    ),
    Employee(
        id=4,
        name="Diana",
        skills=["customer_service", "specialist"],
        hourly_rate=21,
        max_hours=8,
        preferred_shifts=["morning"],
        satisfaction_weight=0.85,
        experience_level="intermediate",
    ),
    Employee(
        id=5,
        name="Eve",
        skills=["cashier", "stocking"],
        hourly_rate=19,
        max_hours=8,
        preferred_shifts=["afternoon"],
        satisfaction_weight=0.75,
        experience_level="junior",
    ),
    Employee(
        id=6,
        name="Frank",
        skills=["customer_service", "trainer"],
        hourly_rate=23,
        max_hours=8,
        preferred_shifts=["full_shift"],
        satisfaction_weight=0.8,
        experience_level="senior",
    ),
]

demand_pattern = [2, 3, 5, 4, 4, 3, 2, 1]
cost_params = {
    "understaff_penalty": 50,
    "overstaff_penalty": 10,
    "break_cost": 5,
}

centaur_scheduler = CentaurScheduler(
    employees=employees,
    demand_pattern=demand_pattern,
    cost_params=cost_params,
    time_horizon=480.0,
)

print("Centaur Scheduler initialized")
print(f"  Employees: {len(employees)}")
print(f"  Demand periods: {len(demand_pattern)}")
print(f"  Peak demand: {max(demand_pattern)} employees")
print(f"  Average demand: {np.mean(demand_pattern):.1f} employees")

## Create managers

Manager priorities influence the review ranking, not the construction rules for the three candidate schedules. Cost is normalized across the candidates before ranking so that currency units do not overwhelm preference and coverage scores. The normalization is relative to this set of candidates; it is not an absolute measure of schedule quality.

In [ ]:
def create_managers() -> List[HumanManager]:
    """Create managers with distinct review priorities and decision styles."""
    return [
        HumanManager(
            id=1,
            name="Sarah Chen",
            department="Operations",
            experience_years=12,
            decision_style="analytical",
            priorities={
                "cost": 0.4,
                "satisfaction": 0.2,
                "coverage": 0.3,
                "robustness": 0.1,
            },
            trust_in_ai=0.8,
            risk_tolerance=0.3,
        ),
        HumanManager(
            id=2,
            name="Michael Rodriguez",
            department="Human Resources",
            experience_years=8,
            decision_style="collaborative",
            priorities={
                "cost": 0.2,
                "satisfaction": 0.4,
                "coverage": 0.3,
                "robustness": 0.1,
            },
            trust_in_ai=0.6,
            risk_tolerance=0.5,
        ),
        HumanManager(
            id=3,
            name="Emily Watson",
            department="Customer Service",
            experience_years=15,
            decision_style="intuitive",
            priorities={
                "cost": 0.1,
                "satisfaction": 0.5,
                "coverage": 0.3,
                "robustness": 0.1,
            },
            trust_in_ai=0.7,
            risk_tolerance=0.6,
        ),
        HumanManager(
            id=4,
            name="David Kim",
            department="Finance",
            experience_years=10,
            decision_style="analytical",
            priorities={
                "cost": 0.5,
                "satisfaction": 0.1,
                "coverage": 0.2,
                "robustness": 0.2,
            },
            trust_in_ai=0.9,
            risk_tolerance=0.2,
        ),
    ]


managers = create_managers()
print("Managers available for schedule review:")
for manager in managers:
    print(
        f"{manager.name} ({manager.department}): "
        f"style={manager.decision_style}, "
        f"experience={manager.experience_years} years"
    )
    print(f"  Priorities: {manager.priorities}")

## Generate, explain, and review candidate schedules

The coverage fraction is also used as a simple descriptive confidence indicator: it is the proportion of demand periods whose stated staffing requirement is met. It is not a probability that a schedule will succeed. The ranked list is decision support only; the manager remains the decision-maker.

In [ ]:
manager = managers[1]
options = centaur_scheduler.generate_schedule_options(manager)

print(f"Schedule review with {manager.name} ({manager.department})")
print(f"Decision style: {manager.decision_style}")
print(f"Priorities: {manager.priorities}")

for index, option in enumerate(options, start=1):
    print(f"\nOption {index}: {option.name}")
    print(f"  ID: {option.id}")
    print(f"  Coverage-based indicator: {option.confidence_score:.1%}")
    print(f"  Explanation: {option.explanation}")
    print(f"  Measures: {'; '.join(option.trade_offs)}")

ranked_options = centaur_scheduler.rank_options(options, manager)
print("\nPriority-based review ranking:")
for rank, (option, score) in enumerate(ranked_options, start=1):
    print(f"  {rank}. {option.name}: review score={score:.3f}")

selected_option = ranked_options[0][0]
print(
    f"\nThe highest-ranked candidate for discussion is "
    f"{selected_option.name}. The manager makes the final selection."
)

## Human feedback and collaborative refinement

A manager can request an explicit shift category for an employee. The scheduler applies the requested shift within that employee's maximum-hour limit, recalculates the schedule measures, and records the interaction. The request is not silently treated as feasible: if it creates a coverage shortfall, the coverage measures and explanation identify that consequence. The history supports later review, but this example does not train or update an AI model.

In [ ]:
human_feedback = {
    "shift_overrides": {2: "full_shift"},
    "comments": "Provide additional coverage during the peak period.",
}

refined_option = centaur_scheduler.apply_human_feedback(
    option=selected_option,
    manager=manager,
    feedback=human_feedback,
)

print(f"Refined candidate: {refined_option.name}")
print(f"Manager feedback: {refined_option.human_feedback}")
print(f"Explanation: {refined_option.explanation}")
print("Updated measures:")
for measure, value in refined_option.objective_scores.items():
    print(f"  {measure}: {value}")
print(f"Recorded interactions: {len(centaur_scheduler.collaboration_history)}")

## Visual comparison

The plots below summarize the managers' stated decision styles, the candidates' measured cost and preference scores, and the managers' stated AI-trust and risk-tolerance values. The candidate measures are calculated from the schedules generated above; manager characteristics come directly from the input records. These descriptive plots do not establish causal relationships or prove that one candidate is universally preferable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    "Human-AI Workforce Scheduling Review",
    fontsize=16,
    fontweight="bold",
)

style_counts = pd.Series(
    [manager.decision_style for manager in managers]
).value_counts()
axes[0, 0].pie(
    style_counts.values,
    labels=style_counts.index,
    autopct="%1.1f%%",
    startangle=90,
)
axes[0, 0].set_title("Manager Decision Styles")

option_names = [option.name for option in options]
cost_values = [option.objective_scores["cost"] for option in options]
axes[0, 1].bar(option_names, cost_values, color=["#2ecc71", "#3498db", "#f39c12"])
axes[0, 1].set_title("Scheduled Labor Cost by Candidate")
axes[0, 1].set_ylabel("Cost (currency units)")
axes[0, 1].tick_params(axis="x", rotation=20)
axes[0, 1].grid(axis="y", alpha=0.3)

satisfaction_values = [
    option.objective_scores["satisfaction"] for option in options
]
coverage_values = [
    option.objective_scores["coverage_fraction"] for option in options
]
for option, cost, satisfaction, coverage in zip(
    options, cost_values, satisfaction_values, coverage_values
):
    axes[1, 0].scatter(cost, satisfaction, s=100, label=option.name)
    axes[1, 0].annotate(
        f"Coverage: {coverage:.0%}",
        (cost, satisfaction),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
axes[1, 0].set_xlabel("Scheduled labor cost (currency units)")
axes[1, 0].set_ylabel("Employee-preference match (fraction)")
axes[1, 0].set_title("Cost and Preference-Match Trade-off")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

trust_scores = [manager.trust_in_ai for manager in managers]
risk_scores = [manager.risk_tolerance for manager in managers]
manager_names = [manager.name for manager in managers]
axes[1, 1].scatter(trust_scores, risk_scores, s=100, color="#6c5ce7")
for trust, risk, name in zip(trust_scores, risk_scores, manager_names):
    axes[1, 1].annotate(
        name,
        (trust, risk),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
axes[1, 1].set_xlabel("Trust in AI (scale: 0 to 1)")
axes[1, 1].set_ylabel("Risk tolerance (scale: 0 to 1)")
axes[1, 1].set_title("Manager Trust and Risk Tolerance")
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The plots summarize candidate measures and manager input attributes.")

## Interpreting the results and the role of this tier

The three candidates expose different operational priorities: lower scheduled labor cost, closer alignment with stated employee shift preferences, and greater staffing surplus. These measures can conflict. A manager should inspect both the headline measures and the period-by-period coverage before selecting a schedule. A high preference-match score does not itself prove that coverage is adequate, and a high coverage surplus may come with additional labor cost.

This human-AI partnership approach is useful when contextual knowledge, accountability, and transparent review matter. Its limitations are equally important: the candidate rules are simplified, the preference score is only a proxy, and manager priorities may omit relevant operational considerations. The recorded interactions are useful for audit and analysis, but they do not constitute adaptive learning in this implementation.

In a broader scheduling system, the same workflow could accommodate additional operational constraints, richer employee preference data, or multiple demand scenarios. Those extensions would require explicit modeling and validation; they are outside the scope of this example.